# Fixed-point bundle solvers: quickstart

This notebook builds one operator on the product domain $\Delta_n^m$ with user-selected
dimensions `m, n`. The same construction covers the simplex case: at `m = 1` the operator
reduces to an operator on $\Delta_n$.

The solver depends on `m`:

- `m = 1`: `VI_simplex`;
- `m > 1`: `VI_simplex_product`.

The notebook then plots the convergence trace.

**Repository note:** the setup cell checks the current directory, its parents, and a
`FiberBundle_VI` subdirectory for a local `solvers/` package. If no local clone is found
(for example in Colab), it clones the repository.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repository(start):
    candidates = [start, *start.parents, start / "FiberBundle_VI"]
    for path in candidates:
        if (path / "solvers").is_dir() and (path / "main.py").is_file():
            return path.resolve()
    return None


root = find_repository(Path.cwd())
if root is None:
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/shb20tsinghua/FiberBundle_VI.git"],
        check=True,
    )
    root = (Path.cwd() / "FiberBundle_VI").resolve()

os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

from solvers import VI_simplex, VI_simplex_product
from main import NeuralNetworkOperator, PayoffTensorOperator

print("repository:", root)
print("Python     ", sys.version.split()[0])
print("NumPy      ", np.__version__)
print("SciPy      ", scipy.__version__)
print("matplotlib ", matplotlib.__version__)


## Custom operator

Both solvers need only an object with a `value_Jaco(x) -> (F, JF)` method. This demo uses

$$
F(x)=b+A x+\tfrac12 x^{\circ 2},
\qquad
JF(x)=A+\operatorname{diag}(x),
$$

where the square is elementwise and `x` is flattened internally.

The class therefore works for both domains:

- simplex: `x` has shape `(n,)`; `F` has shape `(n,)`; `JF` has shape `(n, n)`;
- product of simplex: `x` has shape `(m, n)`; `F` is reshaped to `(m, n)`; `JF` is reshaped to
  `(m, n, m, n)`.


In [ ]:
class CustomOperator:
    """F(x) = b + A x + x**2 / 2, with x flattened internally."""

    def __init__(self, A, b):
        self.A, self.b = A, b

    def value_Jaco(self, x):
        shape = x.shape
        flat = x.ravel()
        values = self.b + self.A @ flat + 0.5 * flat**2
        jacobian = self.A.copy()
        jacobian[np.diag_indices_from(jacobian)] += flat
        return values.reshape(shape), jacobian.reshape(shape + shape)


## Build operator instance

The settings cell chooses `m, n` and the operator:

- `choice = 0`: the quadratic custom operator defined above;
- `choice = 1`: the built-in three-layer neural network with widths `[m*n, m*n, m*n]`;
- `choice = 2`: the built-in low-rank payoff-tensor operator of rank `m*n`.

Every operator preserves the shape of the input point: at `m = 1` it returns `(n,)` and
`(n, n)` and is solved by `VI_simplex`; for `m > 1` it returns `(m, n)` and
`(m, n, m, n)` and is solved by `VI_simplex_product`.


In [ ]:
m, n = 16, 16
SEED = 0
choice = 0

rng = np.random.default_rng(SEED)

if choice == 0:
    A = rng.standard_normal((m * n, m * n))
    b = rng.standard_normal(m * n)
    VI_to_solve = CustomOperator(A, b)
elif choice == 1:
    VI_to_solve = NeuralNetworkOperator(SEED, [m * n, m * n, m * n], 0)
else:
    VI_to_solve = PayoffTensorOperator(SEED, m, n, m * n)


## Solve

The solver is selected by `m`:

- `m == 1`: `VI_simplex` with a point in $\Delta_n$;
- `m > 1`: `VI_simplex_product` with an `(m, n)` point in $\Delta_n^m$.

`GAP_TOL = 1e-8` specifies required precision. `verbose = 1` records the iteration trace. The returned `info` is
`[nit, gap, barrier_sum, G_norm, avoid_n]`. The trace is logged to `data/record.log` for the plot.


In [ ]:
GAP_TOL = 1e-8
trace = Path("data/record.log")
trace.parent.mkdir(parents=True, exist_ok=True)

if m == 1:
    x_init = rng.dirichlet(np.ones(n))
    solution_x, info = VI_simplex(VI_to_solve).solve(
        x_init, gap_tol=GAP_TOL, verbose=1, record_file=str(trace)
    )
else:
    x_init = rng.dirichlet(np.ones(n), size=(m,))
    solution_x, info = VI_simplex_product(VI_to_solve).solve(
        x_init, gap_tol=GAP_TOL, verbose=1, record_file=str(trace)
    )

print(info)

## Convergence

The plot shows the gap on a logarithmic scale. The solver stops once the gap is below
`GAP_TOL`. The title reflects the selected domain.


In [ ]:
rows = [line.split("|") for line in trace.read_text(encoding="utf-8").splitlines()[1:]]
nits = np.array([int(row[1]) for row in rows])
gaps = np.array([float(row[2]) for row in rows])

fig, ax = plt.subplots(figsize=(6, 4), dpi=140)
ax.semilogy(nits, gaps)
ax.set_title(
    f"Simplex VI (dim={n})" if m == 1
    else f"Product-of-simplex VI (dim={m} × {n})"
)
ax.set_xlabel("iteration")
ax.set_ylabel("gap")
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()
